# ADD 2026 Track 2 — MOSS-Audio 8B D006-conditioned LoRA

This notebook is derived from the E004 Rank-32 path in `03_MOSS_Audio_8B_LoRA_Colab.ipynb`. It keeps every E004 training hyperparameter fixed and changes only the text condition: train gold labels are included in train prompts, and D006 predicted labels are included in dev prompts. Generated responses are evaluated without replacing or appending `<Decision>`.

## 1. Configuration

Run from top to bottom in a Colab GPU runtime. The original E004 adapter and outputs use different Drive paths and are never overwritten.

In [ ]:
from pathlib import Path

PROJECT_REPO = "https://github.com/youhan200203/ADD2026-track2-baseline.git"
PROJECT_DIR = Path("/content/ADD2026-track2-baseline")
MOSS_REPO = Path("/content/MOSS-Audio")
MODEL_ID = "OpenMOSS-Team/MOSS-Audio-8B-Instruct"
MODEL_DIR = Path("/content/MOSS-Audio-8B-Instruct")
DATASET_FILE_ID = "1QQIEf5lE5I5dIpKrnxGW8noj71zxZmd8"
DATA_ARCHIVE = Path("/content/track2_training_datasets.zip")
DATA_ROOT = Path("/content/track2_training_datasets")
MOSS_DATA_DIR = PROJECT_DIR / "data/moss"
DRIVE_ROOT = Path("/content/drive/MyDrive/ADD2026/moss_audio_8b")
RUN_NAME = "label_conditioned_goldtrain_d006dev_r32_epoch1_v1"
RUN_DIR = DRIVE_ROOT / RUN_NAME
MAX_LEN_CAP = 4096
MAX_NEW_TOKENS = 2048
INFERENCE_BATCH_SIZE = 1
PER_DEVICE_TRAIN_BATCH_SIZE = 1
GRAD_ACCUMULATION = 8
LEARNING_RATE = 2e-4
SEED = 42
PROMPT_VERSION = 1

E004_REFERENCE = {
    "balanced_accuracy": 0.9975,
    "bert_real": 0.906856,
    "bert_fake": 0.899379,
    "mean_bertscore_f1": 0.903118,
    "final_score_100": 90.14,
}

## 2. Runtime, code, data, and model preparation

These cells are copied from the common E004 setup path.

In [ ]:
import importlib.util, shutil, sys
from importlib.metadata import version as package_version
required_modules = ["accelerate", "gdown", "huggingface_hub", "librosa", "peft", "scipy", "soundfile", "torch", "transformers"]
missing_modules = [name for name in required_modules if importlib.util.find_spec(name) is None]
missing_commands = [name for name in ("git",) if shutil.which(name) is None]
if missing_modules or missing_commands:
    raise RuntimeError({"missing_modules": missing_modules, "missing_commands": missing_commands})
import torch
if not torch.cuda.is_available():
    raise RuntimeError("GPU runtime is required")
print({"python": sys.version.split()[0], "torch": torch.__version__, "transformers_before_setup": package_version("transformers"), "gpu": torch.cuda.get_device_name(0)})

In [ ]:
!python -m pip uninstall -y torchao
!python -m pip install -q transformers==4.57.1
!python -c "import transformers; assert transformers.__version__ == '4.57.1'; print('transformers', transformers.__version__)"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
RUN_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
if PROJECT_DIR.is_dir():
    print(f"[reuse] {PROJECT_DIR}")
else:
    !git clone {PROJECT_REPO} {PROJECT_DIR}
if MOSS_REPO.is_dir():
    print(f"[reuse] {MOSS_REPO}")
else:
    !git clone https://github.com/OpenMOSS/MOSS-Audio.git {MOSS_REPO}
for required in ("prepare_moss_data.py", "prepare_data.py", "moss_train.py", "moss_infer.py", "eval_add2026.py"):
    if not (PROJECT_DIR / required).is_file():
        raise FileNotFoundError(PROJECT_DIR / required)
if not (MOSS_REPO / "finetune/finetune.py").is_file():
    raise FileNotFoundError(MOSS_REPO / "finetune/finetune.py")

In [ ]:
import gdown, os, zipfile
if not all((DATA_ROOT / name).is_dir() for name in ("train", "dev", "labels")):
    if not DATA_ARCHIVE.is_file():
        gdown.download(id=DATASET_FILE_ID, output=DATA_ARCHIVE.as_posix(), quiet=False)
    if not zipfile.is_zipfile(DATA_ARCHIVE):
        raise RuntimeError(f"Downloaded dataset is not a ZIP: {DATA_ARCHIVE}")
    with zipfile.ZipFile(DATA_ARCHIVE) as archive:
        archive.extractall("/content")
if not all((DATA_ROOT / name).is_dir() for name in ("train", "dev", "labels")):
    raise RuntimeError(f"Expected train, dev, labels under {DATA_ROOT}")
project_data = PROJECT_DIR / "data"
project_data.mkdir(exist_ok=True)
for name in ("train", "dev", "labels"):
    target, source = project_data / name, DATA_ROOT / name
    if target.is_symlink():
        if target.resolve() != source.resolve():
            raise RuntimeError(f"Unexpected symlink: {target}")
    elif target.exists():
        raise RuntimeError(f"Refusing to replace: {target}")
    else:
        os.symlink(source, target, target_is_directory=True)
print({name: len(list((DATA_ROOT / name).glob("*.wav"))) for name in ("train", "dev")})

In [ ]:
from huggingface_hub import snapshot_download
snapshot_download(repo_id=MODEL_ID, local_dir=MODEL_DIR)
print(f"[ready] {MODEL_DIR}")

In [ ]:
os.chdir(PROJECT_DIR)
!python -u prepare_moss_data.py --data-root data --output-dir {MOSS_DATA_DIR} --overfit-per-class 4
!python -u prepare_data.py

## 3. Build label-conditioned train and dev prompts

Train prompts use each train sample's gold label. Dev prompts use only the D006 prediction; dev reference labels and Evidence are never placed in the generation prompt. The original audio paths and assistant XML targets are unchanged.

In [ ]:
import copy, hashlib, json, re
from google.colab import files

BASE_TRAIN_MANIFEST = MOSS_DATA_DIR / "train.jsonl"
BASE_DEV_MANIFEST = MOSS_DATA_DIR / "dev.jsonl"
CONDITIONED_TRAIN_MANIFEST = MOSS_DATA_DIR / f"train_label_conditioned_v{PROMPT_VERSION}.jsonl"
D006_DEV_PROMPT_MANIFEST = MOSS_DATA_DIR / f"dev_d006_conditioned_v{PROMPT_VERSION}.jsonl"
D006_CANDIDATES = [
    PROJECT_DIR / "detector_predictions_20261119/D006/dev.jsonl",
    Path("/content/D006_dev.jsonl"),
]

def read_jsonl(path):
    return [json.loads(line) for line in Path(path).read_text(encoding="utf-8-sig").splitlines() if line.strip()]

def write_jsonl(path, rows):
    with Path(path).open("w", encoding="utf-8", newline="\n") as stream:
        for row in rows:
            stream.write(json.dumps(row, ensure_ascii=False) + "\n")

def sha256_file(path, chunk_size=8 * 2**20):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()

d006_path = next((path for path in D006_CANDIDATES if path.is_file()), None)
if d006_path is None:
    print("Upload D006 dev.jsonl", flush=True)
    uploaded = files.upload()
    if "dev.jsonl" not in uploaded:
        raise FileNotFoundError("D006 dev.jsonl was not uploaded")
    d006_path = Path("/content/dev.jsonl")
print(f"[D006] {d006_path}")

In [ ]:
SYSTEM_PROMPT = "You are a senior audio forensics scientist specializing in speech deepfake detection."

def label_conditioned_prompt(label):
    normalized = str(label).strip().lower()
    if normalized not in {"real", "fake"}:
        raise ValueError(f"Invalid condition label: {label}")
    display_label = normalized.capitalize()
    return (
        f"{SYSTEM_PROMPT}\n\n"
        f"The authenticity label of this audio is provided as {display_label}. "
        f"Treat {display_label} as a fixed input condition, not a decision to be independently inferred. "
        "Analyze this audio using observable acoustic evidence and produce a structured XML forensic report "
        "covering spectral, prosodic, pronunciation, noise, and other acoustic anomalies. "
        "Do not mention that the label was provided, and keep every description grounded in the audio. "
        f"The final XML decision must be <Decision>{display_label}</Decision>."
    )

base_train_rows = read_jsonl(BASE_TRAIN_MANIFEST)
base_dev_rows = read_jsonl(BASE_DEV_MANIFEST)
if (len(base_train_rows), len(base_dev_rows)) != (8600, 2000):
    raise RuntimeError(f"Expected 8600/2000 rows, got {len(base_train_rows)}/{len(base_dev_rows)}")

conditioned_train_rows = []
train_label_counts = {"real": 0, "fake": 0}
for source_row in base_train_rows:
    row = copy.deepcopy(source_row)
    label = str(row.get("label", "")).lower()
    if label not in train_label_counts:
        raise ValueError(f"Invalid train label for {row.get('id')}: {label}")
    user_text = [message for message in row["conversation"] if message.get("role") == "user" and message.get("message_type") == "text"]
    assistant_text = [message for message in row["conversation"] if message.get("role") == "assistant" and message.get("message_type") == "text"]
    if len(user_text) != 1 or len(assistant_text) != 1:
        raise ValueError(f"Unexpected conversation structure: {row.get('id')}")
    target_decisions = re.findall(r"<Decision>\s*(real|fake)\s*</Decision>", assistant_text[0]["content"], re.I)
    if len(target_decisions) != 1 or target_decisions[0].lower() != label:
        raise ValueError(f"Target Decision does not match label: {row.get('id')}")
    user_text[0]["content"] = label_conditioned_prompt(label)
    conditioned_train_rows.append(row)
    train_label_counts[label] += 1
write_jsonl(CONDITIONED_TRAIN_MANIFEST, conditioned_train_rows)

d006_rows = read_jsonl(d006_path)
d006_by_id = {}
for row in d006_rows:
    file_id = row.get("file_id")
    label = str(row.get("label", "")).lower()
    if not file_id or file_id in d006_by_id or label not in {"real", "fake"}:
        raise ValueError(f"Invalid or duplicate D006 row: {row}")
    d006_by_id[file_id] = {"label": label, "score": row.get("score")}
dev_ids = [row["id"] for row in base_dev_rows]
if len(d006_by_id) != 2000 or set(d006_by_id) != set(dev_ids):
    raise RuntimeError(f"D006/dev ID mismatch: predictions={len(d006_by_id)}, dev={len(dev_ids)}, missing={sorted(set(dev_ids)-set(d006_by_id))[:5]}, unexpected={sorted(set(d006_by_id)-set(dev_ids))[:5]}")
conditioned_dev_rows = [
    {"id": file_id, "conversation": [{"role": "user", "message_type": "text", "content": label_conditioned_prompt(d006_by_id[file_id]["label"])}]}
    for file_id in dev_ids
]
write_jsonl(D006_DEV_PROMPT_MANIFEST, conditioned_dev_rows)
d006_label_counts = {label: sum(item["label"] == label for item in d006_by_id.values()) for label in ("real", "fake")}
manifest_summary = {
    "prompt_version": PROMPT_VERSION,
    "train_rows": len(conditioned_train_rows), "train_label_counts": train_label_counts,
    "dev_rows": len(conditioned_dev_rows), "dev_condition_source": d006_path.as_posix(),
    "d006_label_counts": d006_label_counts,
    "train_manifest_sha256": sha256_file(CONDITIONED_TRAIN_MANIFEST),
    "dev_prompt_manifest_sha256": sha256_file(D006_DEV_PROMPT_MANIFEST),
    "d006_sha256": sha256_file(d006_path),
}
MANIFEST_SUMMARY_PATH = RUN_DIR / "conditioned_manifest_summary.json"
MANIFEST_SUMMARY_PATH.write_text(json.dumps(manifest_summary, indent=2), encoding="utf-8")
print(json.dumps(manifest_summary, indent=2))
print("\n[Real prompt]\n" + label_conditioned_prompt("real"))
print("\n[Fake prompt]\n" + label_conditioned_prompt("fake"))

## 4. Conditional sequence-length validation

This repeats E004's exact length accounting on the modified train prompt and displays progress with an ETA.

In [ ]:
import math, numpy as np, soundfile as sf, time
from transformers import AutoTokenizer

def format_eta(seconds):
    if not math.isfinite(seconds) or seconds < 0:
        return "unknown"
    return time.strftime("%H:%M:%S", time.gmtime(seconds))

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, trust_remote_code=True)
train_lengths = []
length_started = time.monotonic()
for index, row in enumerate(conditioned_train_rows, start=1):
    audio, prompt, answer = [part["content"] for part in row["conversation"]]
    info = sf.info(audio)
    audio_tokens = math.ceil((info.frames / info.samplerate) * 12.5)
    text_tokens = len(tokenizer.encode(prompt, add_special_tokens=False)) + len(tokenizer.encode(answer, add_special_tokens=False))
    train_lengths.append(23 + audio_tokens + text_tokens)
    if index == 1 or index % 500 == 0 or index == len(conditioned_train_rows):
        elapsed = time.monotonic() - length_started
        rate = index / max(elapsed, 1e-9)
        print(f"[lengths] {index}/{len(conditioned_train_rows)} | {rate:.1f} samples/s | ETA {format_eta((len(conditioned_train_rows)-index)/max(rate, 1e-9))}", flush=True)
train_lengths_array = np.asarray(train_lengths)
length_report = {name: float(np.percentile(train_lengths_array, value)) for name, value in {"p50": 50, "p95": 95, "p99": 99, "p99_5": 99.5}.items()}
length_report["max"] = int(train_lengths_array.max())
MAX_LEN = min(MAX_LEN_CAP, max(512, math.ceil(length_report["max"] / 128) * 128))
length_report["selected_max_len"] = MAX_LEN
length_report["truncated_samples"] = int((train_lengths_array > MAX_LEN).sum())
LENGTH_REPORT_PATH = RUN_DIR / "sequence_length_report.json"
LENGTH_REPORT_PATH.write_text(json.dumps(length_report, indent=2), encoding="utf-8")
print(json.dumps(length_report, indent=2))
del tokenizer

## 5. E004-controlled Rank-32 LoRA training

The data count and all optimizer, LoRA, scheduler, checkpoint, precision, and seed settings are identical to E004. Only `data_path` points to the gold-label-conditioned train manifest.

In [ ]:
TRAIN_ROWS = len(conditioned_train_rows)
if TRAIN_ROWS != 8600:
    raise RuntimeError(f"Expected 8600 train rows, got {TRAIN_ROWS}")
UPDATES_PER_EPOCH = math.ceil(TRAIN_ROWS / (PER_DEVICE_TRAIN_BATCH_SIZE * GRAD_ACCUMULATION))
WARMUP_1E = math.ceil(UPDATES_PER_EPOCH * 0.03)
PROFILE_PATH = RUN_DIR / "profile.json"
print({"train_rows": TRAIN_ROWS, "updates_per_epoch": UPDATES_PER_EPOCH, "warmup_steps": WARMUP_1E, "max_len": MAX_LEN})

In [ ]:
def resolve_adapter(output_dir, profile_path):
    if (output_dir / "adapter_config.json").is_file():
        return output_dir
    if profile_path.is_file():
        profile = json.loads(profile_path.read_text(encoding="utf-8"))
        latest = profile.get("latest_checkpoint")
        if latest and (Path(latest) / "adapter_config.json").is_file():
            return Path(latest)
    checkpoints = sorted(output_dir.glob("checkpoint-*"), key=lambda path: int(path.name.rsplit("-", 1)[-1]) if path.name.rsplit("-", 1)[-1].isdigit() else -1)
    return next((path for path in reversed(checkpoints) if (path / "adapter_config.json").is_file()), None)

adapter_before = resolve_adapter(RUN_DIR, PROFILE_PATH)
completed_before = False
if adapter_before is not None and PROFILE_PATH.is_file():
    existing_profile = json.loads(PROFILE_PATH.read_text(encoding="utf-8"))
    completed_before = int(existing_profile.get("global_step", 0)) >= UPDATES_PER_EPOCH
if completed_before:
    print(f"[skip] completed: {adapter_before}")
else:
    !PYTHONPATH={MOSS_REPO} PYTHONUNBUFFERED=1 python -u moss_train.py \
      --moss-repo {MOSS_REPO} --profile-json {PROFILE_PATH} --resume-from-checkpoint auto --require-lora-gradient \
      --model_dir {MODEL_DIR} --data_path {CONDITIONED_TRAIN_MANIFEST} \
      --output_dir {RUN_DIR} --use_lora --lora_rank 32 --lora_alpha 64 --attn_implementation sdpa \
      --bf16 True --tf32 True --gradient_checkpointing True --per_device_train_batch_size {PER_DEVICE_TRAIN_BATCH_SIZE} \
      --gradient_accumulation_steps {GRAD_ACCUMULATION} --learning_rate {LEARNING_RATE} --num_train_epochs 1 \
      --lr_scheduler_type cosine --warmup_steps {WARMUP_1E} --logging_steps 10 --logging_first_step True \
      --save_strategy steps --save_steps 200 --save_total_limit 2 --report_to none \
      --remove_unused_columns False --dataloader_num_workers 2 --max_len {MAX_LEN} --seed {SEED}

ADAPTER_DIR = resolve_adapter(RUN_DIR, PROFILE_PATH)
if ADAPTER_DIR is None or not PROFILE_PATH.is_file():
    raise FileNotFoundError(f"No completed adapter/profile found under {RUN_DIR}")
profile = json.loads(PROFILE_PATH.read_text(encoding="utf-8"))
if int(profile.get("global_step", 0)) < UPDATES_PER_EPOCH:
    raise RuntimeError(f"Training incomplete: global_step={profile.get('global_step')} expected>={UPDATES_PER_EPOCH}")
if float(profile.get("max_lora_gradient_norm", 0.0)) <= 0:
    raise RuntimeError("LoRA gradient validation failed")
print({"adapter": ADAPTER_DIR.as_posix(), "global_step": profile["global_step"], "max_lora_gradient_norm": profile["max_lora_gradient_norm"]})

## 6. D006-conditioned dev generation

Each D006 label is present in the prompt before the first generated token. `moss_infer.py` writes the model response unchanged; this notebook performs no Decision postprocessing.

In [ ]:
if importlib.util.find_spec("bert_score") is None:
    !python -m pip install -q bert-score
if importlib.util.find_spec("bert_score") is None:
    raise RuntimeError("bert-score installation failed")
PREDICTION_PATH = DRIVE_ROOT / f"{RUN_NAME}_dev_bs{INFERENCE_BATCH_SIZE}.jsonl"
METRICS_PATH = DRIVE_ROOT / f"{RUN_NAME}_dev_metrics_bs{INFERENCE_BATCH_SIZE}.json"
ADHERENCE_PATH = DRIVE_ROOT / f"{RUN_NAME}_dev_condition_adherence.json"
MISMATCH_PATH = DRIVE_ROOT / f"{RUN_NAME}_dev_condition_mismatches.jsonl"
print({"predictions": str(PREDICTION_PATH), "metrics": str(METRICS_PATH)})

In [ ]:
if not (ADAPTER_DIR / "adapter_config.json").is_file():
    raise FileNotFoundError(ADAPTER_DIR / "adapter_config.json")
!python -u moss_infer.py --moss-repo {MOSS_REPO} --model-dir {MODEL_DIR} --adapter-dir {ADAPTER_DIR} --audio-dir data/dev --prompt-manifest {D006_DEV_PROMPT_MANIFEST} --output {PREDICTION_PATH} --max-new-tokens {MAX_NEW_TOKENS} --batch-size {INFERENCE_BATCH_SIZE} --merge-lora --resume

## 7. Condition adherence and official evaluation

Adherence compares the model-generated `<Decision>` with the D006 label. It does not alter the response. Official evaluation then scores that untouched response.

In [ ]:
prediction_rows = read_jsonl(PREDICTION_PATH)
prediction_by_id = {}
for row in prediction_rows:
    file_id = row.get("id")
    if not file_id or file_id in prediction_by_id:
        raise ValueError(f"Missing or duplicate prediction id: {file_id}")
    prediction_by_id[file_id] = row
if len(prediction_by_id) != 2000 or set(prediction_by_id) != set(d006_by_id):
    raise RuntimeError(f"Prediction ID mismatch: predictions={len(prediction_by_id)}, D006={len(d006_by_id)}")

def generated_decision(response):
    matches = re.findall(r"<Decision>\s*(real|fake)\s*</Decision>", response or "", re.I)
    return matches[0].lower() if len(matches) == 1 else None

adherence_rows = []
for file_id in dev_ids:
    condition = d006_by_id[file_id]["label"]
    generated = generated_decision(prediction_by_id[file_id].get("response"))
    adherence_rows.append({
        "file_id": file_id, "condition_label": condition, "generated_decision": generated,
        "matches_condition": generated == condition, "d006_score": d006_by_id[file_id]["score"],
    })
gold_label_by_id = {row["id"]: str(row["label"]).lower() for row in base_dev_rows}
if set(gold_label_by_id) != set(d006_by_id):
    raise RuntimeError("Dev gold-label IDs do not match D006 IDs")
d006_class_results = {}
for label in ("real", "fake"):
    label_ids = [file_id for file_id, gold in gold_label_by_id.items() if gold == label]
    correct = sum(d006_by_id[file_id]["label"] == label for file_id in label_ids)
    d006_class_results[label] = {"total": len(label_ids), "correct": correct, "recall": correct/len(label_ids)}
d006_balanced_accuracy = 0.5 * (d006_class_results["real"]["recall"] + d006_class_results["fake"]["recall"])
condition_adherence = {
    "total": len(adherence_rows),
    "matched": sum(row["matches_condition"] for row in adherence_rows),
    "rate": sum(row["matches_condition"] for row in adherence_rows) / len(adherence_rows),
    "invalid_or_multiple_decisions": sum(row["generated_decision"] is None for row in adherence_rows),
    "by_condition": {
        label: {
            "total": sum(row["condition_label"] == label for row in adherence_rows),
            "matched": sum(row["condition_label"] == label and row["matches_condition"] for row in adherence_rows),
        }
        for label in ("real", "fake")
    },
    "d006_against_dev_gold": {"real": d006_class_results["real"], "fake": d006_class_results["fake"], "balanced_accuracy": d006_balanced_accuracy},
}
for label in ("real", "fake"):
    values = condition_adherence["by_condition"][label]
    values["rate"] = values["matched"] / values["total"] if values["total"] else 0.0
ADHERENCE_PATH.write_text(json.dumps(condition_adherence, indent=2), encoding="utf-8")
write_jsonl(MISMATCH_PATH, [row for row in adherence_rows if not row["matches_condition"]])
print(f"[D006 acc] real={d006_class_results['real']['recall']:.4f} ({d006_class_results['real']['correct']}/{d006_class_results['real']['total']}), fake={d006_class_results['fake']['recall']:.4f} ({d006_class_results['fake']['correct']}/{d006_class_results['fake']['total']}), balanced={d006_balanced_accuracy:.4f}")
print(f"[condition] D006 label adherence={condition_adherence['rate']:.4f} ({condition_adherence['matched']}/{condition_adherence['total']}), invalid={condition_adherence['invalid_or_multiple_decisions']}")
print(json.dumps(condition_adherence["by_condition"], indent=2))

In [ ]:
def metrics_are_current(metrics_path, prediction_path):
    if not metrics_path.is_file() or metrics_path.stat().st_mtime_ns < prediction_path.stat().st_mtime_ns:
        return False
    try:
        metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
    except Exception:
        return False
    expected = {
        "checkpoint": prediction_path.as_posix(), "subset_total": 2000,
        "prediction_file_rows": 2000, "unique_predictions": 2000,
        "matched_predictions": 2000, "missing_predictions": 0, "bertscore_pairs": 2000,
    }
    return all(metrics.get(key) == value for key, value in expected.items())

if not metrics_are_current(METRICS_PATH, PREDICTION_PATH):
    !python -u eval_add2026.py {PREDICTION_PATH} --subset data/swift/dev.jsonl --out-json {METRICS_PATH} --bert-batch-size 32 --bert-device cuda:0
    if not metrics_are_current(METRICS_PATH, PREDICTION_PATH):
        raise RuntimeError(f"Evaluation did not produce valid current metrics: {METRICS_PATH}")
else:
    print(f"[skip] current metrics already exist: {METRICS_PATH}")

## 8. E004 comparison and saved summary

In [ ]:
metrics = json.loads(METRICS_PATH.read_text(encoding="utf-8"))
real, fake = metrics["real"], metrics["fake"]
conditional_result = {
    "balanced_accuracy": metrics["balanced_accuracy"],
    "bert_real": real["mean_bertscore_f1_all"],
    "bert_fake": fake["mean_bertscore_f1_all"],
    "mean_bertscore_f1": metrics["mean_bertscore_f1"],
    "final_score_100": metrics["final_score_100"],
}
print(f"[acc] real={real['recall']:.4f} ({real['correct']}/{real['total']}), fake={fake['recall']:.4f} ({fake['correct']}/{fake['total']}), balanced={metrics['balanced_accuracy']:.4f}")
print(f"[bert] real={real['mean_bertscore_f1_all']:.6f}, fake={fake['mean_bertscore_f1_all']:.6f}, mean={metrics['mean_bertscore_f1']:.6f}, mean_100={100*metrics['mean_bertscore_f1']:.4f}")
print(f"[joint] real={real['joint']:.4f}, fake={fake['joint']:.4f}, J={metrics['joint_score']:.4f}, Score={metrics['final_score_100']:.2f}")
print(f"[condition] D006 label adherence={condition_adherence['rate']:.4f} ({condition_adherence['matched']}/{condition_adherence['total']})")

comparison = [
    {"experiment": "E004", **E004_REFERENCE, "condition_adherence": None},
    {"experiment": RUN_NAME, **conditional_result, "condition_adherence": condition_adherence["rate"]},
]
try:
    import pandas as pd
    display(pd.DataFrame(comparison).style.format({
        "balanced_accuracy": "{:.4f}", "bert_real": "{:.6f}", "bert_fake": "{:.6f}",
        "mean_bertscore_f1": "{:.6f}", "final_score_100": "{:.2f}", "condition_adherence": "{:.4f}",
    }, na_rep="-"))
except ImportError:
    print(json.dumps(comparison, indent=2))

summary = {
    "experiment": RUN_NAME, "derived_from": "E004 Rank-32 epoch-1",
    "controlled_settings": {
        "model": MODEL_ID, "lora_rank": 32, "lora_alpha": 64, "epochs": 1,
        "learning_rate": LEARNING_RATE, "scheduler": "cosine", "warmup_steps": WARMUP_1E,
        "per_device_train_batch_size": PER_DEVICE_TRAIN_BATCH_SIZE, "gradient_accumulation": GRAD_ACCUMULATION,
        "gradient_checkpointing": True, "seed": SEED, "max_len": MAX_LEN,
    },
    "single_changed_factor": "gold train label in train prompt; D006 predicted label in dev prompt",
    "decision_postprocessing": False,
    "manifest_summary": manifest_summary, "condition_adherence": condition_adherence,
    "E004_reference": E004_REFERENCE, "conditional_result": conditional_result,
    "adapter_dir": ADAPTER_DIR.as_posix(), "predictions": PREDICTION_PATH.as_posix(), "metrics": METRICS_PATH.as_posix(),
}
SUMMARY_PATH = RUN_DIR / "experiment_summary.json"
SUMMARY_PATH.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"[saved] {SUMMARY_PATH}")